# Report Black Friday week on a budget

*Data Analysis in Python · Topic 13.2*

Your team wants Black Friday week 2025's purchases and revenue by channel group, and your manager has asked you to keep every query under 50 MB. Measure the query before you run it, and run it with a limit.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. In the parameters cell, set `PROJECT_ID` to your own Google Cloud project's ID, as in topic 13.1. Run the parameters cell, then the login cell.
2. In the first cell marked ✏️, write the function `dry_run_bytes(sql)`: the bytes BigQuery would process for `sql`, from a dry run without the query cache. Then create `bytes_star`: the dry run's bytes for every column of `TABLE` from `START_DATE` to `END_DATE`, with `SELECT *` and `_TABLE_SUFFIX`.
3. In the second cell marked ✏️, write the query `sql`: topic 13.1.'s channel group query, for `START_DATE` to `END_DATE`, with the columns `channel_group`, `purchases` and `revenue`, sorted by revenue, largest first. Create `bytes_query`: its dry run's bytes. It should be under `BUDGET`.
4. In the last cell marked ✏️, create the `bf_channels` table: the query's result, run with `maximum_bytes_billed=BUDGET`.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `bytes_star` is 554,391,054, over the budget. `bytes_query` is 14,481,987, under it. The `bf_channels` table has 9 channel groups, with 2,641 purchases and $171,192.19 of revenue. Email has the most revenue, $42,221.18.

## Parameters

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

PROJECT_ID = ""  # Your own Google Cloud project's ID: the queries run there
TABLE = "simmer-python.simmer_store.events_*"  # Simmer Store's GA4 export, one table per day
START_DATE, END_DATE = "20251124", "20251130"  # Black Friday week 2025, as _TABLE_SUFFIX values
BUDGET = 50_000_000  # 50 MB: the most any query may process

In [ ]:
# Log in to Google as yourself. Run it, choose your account and Allow.
auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)

## The dry run

In [ ]:
# ✏️ Write dry_run_bytes() and create bytes_star


## The query

In [ ]:
# ✏️ Create sql and bytes_query


## The result

In [ ]:
# ✏️ Create bf_channels


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    message = f"{type(error).__name__}: {error}" if error is not None else ""
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif "bytes billed" in message:
        results.append((None, "BigQuery stopped a query over the budget, as maximum_bytes_billed should. Bring the query's scan under BUDGET, then choose Runtime → Restart session and run all."))
    elif "bigquery.jobs.create" in message:
        results.append((None, "BigQuery answered 403: your queries can't run in this project. Set PROJECT_ID to your own project's ID, not simmer-python, then choose Runtime → Restart session and run all."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    project = found.get("PROJECT_ID")
    if not isinstance(project, str) or not project.strip():
        results.append((False, "PROJECT_ID is empty. Set it to your own Google Cloud project's ID, as in topic 13.1."))

    budget = found.get("BUDGET", 50_000_000)
    bytes_star = found.get("bytes_star")
    if not callable(found.get("dry_run_bytes")):
        results.append((False, "dry_run_bytes should be a function: client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)).total_bytes_processed."))
    elif not pd.api.types.is_number(bytes_star):
        results.append((False, "bytes_star should be a number: dry_run_bytes() of SELECT * for Black Friday week."))
    elif bytes_star > 5_000_000_000:
        results.append((False, f"bytes_star is {bytes_star:,}: every column of every table. Pick Black Friday week's tables with WHERE _TABLE_SUFFIX BETWEEN '{found.get('START_DATE')}' AND '{found.get('END_DATE')}'."))
    elif bytes_star < budget:
        results.append((False, f"bytes_star is {bytes_star:,}, under the budget. It should be SELECT *, every column, for the week."))
    else:
        results.append((True, f"SELECT * for the week would process {bytes_star:,} bytes, {bytes_star / budget:.0f} times the budget."))

    bytes_query = found.get("bytes_query")
    if not pd.api.types.is_number(bytes_query):
        results.append((False, "bytes_query should be a number: dry_run_bytes(sql)."))
    elif bytes_query > budget and bytes_query > 700_000_000:
        results.append((False, f"bytes_query is {bytes_query:,}, over the budget of {budget:,}. The query reads every day's table: pick the week's tables with _TABLE_SUFFIX instead of event_date."))
    elif bytes_query > budget:
        results.append((False, f"bytes_query is {bytes_query:,}, over the budget of {budget:,}. Name only the columns you need, and pick the week's tables with _TABLE_SUFFIX."))
    else:
        results.append((True, f"The query would process {bytes_query:,} bytes, under the budget of {budget:,}."))

    table = found.get("bf_channels")
    columns = ["channel_group", "purchases", "revenue"]
    ran_with_limit = isinstance(inputs, list) and any("maximum_bytes_billed" in source for source in inputs[1:-1])
    if not isinstance(table, pd.DataFrame) or [column for column in columns if column not in table.columns]:
        results.append((False, f"bf_channels should be a DataFrame with the columns {', '.join(columns)}."))
    elif int(table["purchases"].sum()) > 100000:
        results.append((False, "The table counts every event, not purchases. Keep only the purchases: event_name = 'purchase'."))
    elif len(table) != 9:
        results.append((False, f"The bf_channels table should have 9 rows, one per channel group. It has {len(table)}."))
    elif int(table["purchases"].sum()) != 2641 or abs(float(table["revenue"].sum()) - 171192.19) > 0.05:
        results.append((False, f"Black Friday week has 2,641 purchases and $171,192.19 of revenue. The table has {int(table['purchases'].sum()):,} and ${float(table['revenue'].sum()):,.2f}. Check the dates and that you sum ecommerce.purchase_revenue."))
    elif not ran_with_limit:
        results.append((False, "Run the query with a limit: client.query(sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=BUDGET))."))
    elif table.iloc[0]["channel_group"] != "Email":
        results.append((False, "Sort by revenue, largest first: ORDER BY revenue DESC."))
    else:
        results.append((True, "9 channel groups, with 2,641 purchases and $171,192.19 of revenue. Email has the most, $42,221.18."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def dry_run_bytes(sql):
    """Return the bytes BigQuery would scan for sql, without running it."""
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=config).total_bytes_processed


bytes_star = dry_run_bytes(f"SELECT * FROM `{TABLE}` WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}'")
bytes_star
```

```python
sql = f"""
SELECT
  session_traffic_source_last_click.cross_channel_campaign.default_channel_group AS channel_group,
  COUNT(*) AS purchases,
  ROUND(SUM(ecommerce.purchase_revenue), 2) AS revenue
FROM `{TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}' AND event_name = 'purchase'
GROUP BY channel_group
ORDER BY revenue DESC
"""
bytes_query = dry_run_bytes(sql)
bytes_query
```

```python
config = bigquery.QueryJobConfig(maximum_bytes_billed=BUDGET)
bf_channels = client.query(sql, job_config=config).to_dataframe()
bf_channels
```

</details>